# K-EL/01 — Run and view the FreeCAD macro

This notebook is a companion to `K_EL01_Split_Yoke_Concept.FCMacro`. It lets you **inspect the macro source, view an interactive 3D preview of the supplied STEP model, and (on a computer with FreeCAD installed) launch FreeCAD to execute a working copy of the macro**.

**Important:** an `.FCMacro` is not a standalone ordinary Python script. It imports FreeCAD-specific modules (`FreeCAD`, `Part`, and optionally `MeshPart`) and must execute inside the FreeCAD application. This notebook launches that application from a local Jupyter environment; it cannot run the FreeCAD GUI from a normal hosted Colab runtime.

The macro describes a concept-envelope model, not manufacturing-ready CAD. Dimensions and catalogue interfaces contain assumptions and the model is not strength, tolerance, thermal, collision, or safety validated.

## 1. Locate the macro and STEP file

Put the `.ipynb`, `.FCMacro`, and `.step` files in the same folder for the simplest setup. On Google Colab, the cell below asks you to upload the macro and STEP file. On local Jupyter, set `WORKDIR` to the folder where you saved the files if the automatic lookup misses them.

In [ ]:
from pathlib import Path
import os, sys, platform, shutil, subprocess, time, re

# For local Jupyter: change this to the folder containing the downloaded files if needed.
WORKDIR = Path.cwd()
MACRO_NAME = "K_EL01_Split_Yoke_Concept.FCMacro"
STEP_NAME = "K_EL01_Split_Yoke_Concept.step"

# Hosted Colab: upload both files. Local Jupyter: skip upload and use files beside the notebook.
try:
    import google.colab  # type: ignore
    from google.colab import files
    print("Google Colab detected. Upload the .FCMacro and .step files in the file picker.")
    uploaded = files.upload()
    for name, data in uploaded.items():
        Path(name).write_bytes(data)
    WORKDIR = Path.cwd()
except Exception:
    print("Local Jupyter / Python environment detected.")

# Search the notebook folder and the release folder used in the provided package.
def locate_file(filename):
    candidates = [WORKDIR / filename,
                  Path.cwd() / filename,
                  Path("/mnt/data/Kritva_K_EL01_Final_Audited_Release") / filename,
                  Path("/mnt/data/Kritva_K_EL01_Comprehensive_Release") / filename,
                  Path("/mnt/data") / filename]
    for p in candidates:
        if p.exists():
            return p.resolve()
    return None

MACRO_PATH = locate_file(MACRO_NAME)
STEP_PATH = locate_file(STEP_NAME)
print("Working directory:", WORKDIR.resolve())
print("Macro:", MACRO_PATH or "NOT FOUND")
print("STEP:", STEP_PATH or "NOT FOUND")
if MACRO_PATH is None:
    raise FileNotFoundError(f"Cannot find {MACRO_NAME}. Put it beside this notebook or set MACRO_PATH manually.")
if STEP_PATH is None:
    print("STEP file not found yet. The source can still be inspected, but a 3D STEP preview needs the .step file.")

## 2. Read and inspect the macro source

This displays the actual macro as Python source; it does not execute FreeCAD-specific code inside the notebook kernel.

In [ ]:
from IPython.display import display, Code, Markdown
macro_source = MACRO_PATH.read_text(encoding="utf-8", errors="replace")
print(f"Macro size: {len(macro_source):,} characters; {len(macro_source.splitlines())} lines")
display(Code(macro_source, language="python"))

## 3. Extract the main geometry assumptions

These values are read from the macro source so you can quickly see which dimensions are explicitly assumed in the concept model. They are not independent validation of the design.

In [ ]:
# Show assignment-style constants and design properties directly from the macro text.
constant_pattern = re.compile(r"^([A-Z][A-Z0-9_]+)\s*=\s*(.+?)\s*(?:#.*)?$", re.M)
constants = constant_pattern.findall(macro_source)
print("Geometry / sizing constants found in macro:\n")
for key, value in constants:
    if any(token in key for token in ("LENGTH", "SHAFT", "BEARING", "HOUSING", "MOTOR", "GEAR", "SPACING")):
        print(f"{key:24s} = {value}")
print("\nDesign-basis labels near the bottom of the macro:")
for line in macro_source.splitlines():
    if 'props=[' in line or any(tag in line for tag in ('StaticTorque_Nm', 'TargetTorque_Nm', 'TargetSpeed_RPM', 'ModelStatus', 'Unresolved')):
        print(line.strip())

## 4. Show an interactive 3D preview from the STEP file

The next cell imports the supplied neutral CAD model using CadQuery/OpenCascade and renders a rotatable Plotly mesh in the notebook. If the libraries are missing in a local environment, install them with `python -m pip install cadquery plotly`.

In [ ]:
if STEP_PATH is None:
    print("No STEP file is available. Download K_EL01_Split_Yoke_Concept.step and put it beside this notebook.")
else:
    try:
        import cadquery as cq
        import plotly.graph_objects as go
        step_wp = cq.importers.importStep(str(STEP_PATH))
        step_shape = step_wp.val()
        # Linear deflection is in model units (mm). Smaller values create a finer, heavier mesh.
        vertices, triangles = step_shape.tessellate(0.8, 0.35)
        xs = [v.x for v in vertices]
        ys = [v.y for v in vertices]
        zs = [v.z for v in vertices]
        ii = [t[0] for t in triangles]
        jj = [t[1] for t in triangles]
        kk = [t[2] for t in triangles]
        fig = go.Figure(data=[go.Mesh3d(
            x=xs, y=ys, z=zs, i=ii, j=jj, k=kk,
            opacity=1.0, flatshading=False,
            lighting=dict(ambient=0.55, diffuse=0.75, specular=0.22, roughness=0.7, fresnel=0.12),
            lightposition=dict(x=1000, y=1000, z=1500),
            name="K-EL/01 concept")])
        fig.update_layout(
            title="K-EL/01 Split-Yoke Actuator — STEP concept preview",
            scene=dict(xaxis_title="X (mm)", yaxis_title="Y (mm)", zaxis_title="Z (mm)",
                       aspectmode="data", camera=dict(eye=dict(x=1.7, y=1.7, z=1.2))),
            margin=dict(l=0, r=0, t=45, b=0), height=700)
        fig.show()
        print(f"STEP preview generated: {len(vertices):,} mesh vertices and {len(triangles):,} triangles.")
        print("This preview comes from the supplied STEP model; it does not prove that the macro executes successfully.")
    except ImportError as exc:
        print("CadQuery or Plotly is missing. Install them using: python -m pip install cadquery plotly")
        raise
    except Exception as exc:
        print("Could not render STEP preview:", repr(exc))
        print("Check that the STEP file is complete and supported by your CadQuery/OpenCascade install.")

## 5. Find FreeCAD on this computer

This section is for **local Jupyter Notebook/JupyterLab on the same computer where FreeCAD is installed**. If you are using Colab, this may not work because the hosted runtime has no desktop FreeCAD GUI. Set `FREECAD_EXECUTABLE` manually if auto-detection does not find it.

In [ ]:
# Optionally set the exact executable path. Examples:
# Windows: r"C:\Program Files\FreeCAD 1.0\bin\FreeCAD.exe"
# macOS:   "/Applications/FreeCAD.app/Contents/MacOS/FreeCAD"
# Linux:   "/usr/bin/freecad"
FREECAD_EXECUTABLE = ""

candidates = []
if FREECAD_EXECUTABLE.strip():
    candidates.append(Path(FREECAD_EXECUTABLE.strip().strip('"')))
for name in ("FreeCAD", "freecad", "freecadcmd", "FreeCADCmd"):
    found = shutil.which(name)
    if found:
        candidates.append(Path(found))
if platform.system() == "Windows":
    for root in (Path(os.environ.get("ProgramFiles", r"C:\Program Files")),
                 Path(os.environ.get("ProgramFiles(x86)", r"C:\Program Files (x86)"))):
        if root.exists():
            candidates.extend(root.glob("FreeCAD*/bin/FreeCAD.exe"))
            candidates.extend(root.glob("FreeCAD*/FreeCAD.exe"))
elif platform.system() == "Darwin":
    candidates.append(Path("/Applications/FreeCAD.app/Contents/MacOS/FreeCAD"))

FREECAD_PATH = next((p.resolve() for p in candidates if p.exists() and p.is_file()), None)
print("Operating system:", platform.platform())
print("FreeCAD executable:", FREECAD_PATH or "NOT FOUND")
if FREECAD_PATH is None:
    print("Install FreeCAD or set FREECAD_EXECUTABLE to its full executable path. The STEP preview above can still be viewed without running the macro.")

## 6. Prepare a run copy that saves beside this notebook and exports a viewing mesh

The original `.FCMacro` is left unchanged. This cell writes a temporary working copy with its save path redirected to the notebook folder and appends an STL export step. That STL can be displayed with the notebook after FreeCAD finishes.

In [ ]:
# Outputs from the FreeCAD run will be written to WORKDIR.
FCSTD_OUTPUT = (WORKDIR / "K_EL01_Split_Yoke_Concept.FCStd").resolve()
STL_OUTPUT = (WORKDIR / "K_EL01_Split_Yoke_Concept_preview.stl").resolve()
RUN_MACRO_PATH = (WORKDIR / "K_EL01_RunAndExport.FCMacro").resolve()

source_for_run = macro_source
old_save_line = 'target=os.path.join(os.path.expanduser("~"),"K_EL01_Split_Yoke_Concept.FCStd")'
new_save_line = f'target = r"{FCSTD_OUTPUT}"'
if old_save_line not in source_for_run:
    raise ValueError("The expected auto-save line was not found in this macro version. No working copy was written.")
source_for_run = source_for_run.replace(old_save_line, new_save_line)
export_tail = f"""
# Notebook companion: export all non-empty Part::Feature shapes as one viewing STL.
try:
    import Mesh, MeshPart
    _combined_mesh = Mesh.Mesh()
    _mesh_count = 0
    for _obj in doc.Objects:
        if getattr(_obj, "TypeId", "") == "Part::Feature" and hasattr(_obj, "Shape") and not _obj.Shape.isNull():
            _one_mesh = MeshPart.meshFromShape(Shape=_obj.Shape, LinearDeflection=0.8, AngularDeflection=0.35, Relative=False)
            _combined_mesh.addMesh(_one_mesh)
            _mesh_count += 1
    if _mesh_count:
        _combined_mesh.write(r"{STL_OUTPUT}")
        print("Notebook STL preview exported from", _mesh_count, "Part::Feature objects to: {STL_OUTPUT}")
    else:
        print("No Part::Feature objects were found for STL export.")
except Exception as _mesh_exc:
    print("STL preview export failed; the FreeCAD document may still have been saved:", str(_mesh_exc))
"""
RUN_MACRO_PATH.write_text(source_for_run + "\n" + export_tail, encoding="utf-8")
print("Original macro (unchanged):", MACRO_PATH)
print("Working macro to execute:", RUN_MACRO_PATH)
print("Expected native FreeCAD file:", FCSTD_OUTPUT)
print("Expected notebook preview mesh:", STL_OUTPUT)

## 7. Run the macro in FreeCAD

Execute the next cell only when you are ready to open FreeCAD. It launches FreeCAD in a separate desktop window and returns control to the notebook; the macro runs inside FreeCAD. You may see a normal FreeCAD window while it creates the assembly. If it errors, read the FreeCAD Report view / Python console.

In [ ]:
if FREECAD_PATH is None:
    print("FreeCAD was not found. Set FREECAD_EXECUTABLE in Section 5, then rerun Sections 5–7.")
else:
    print("Launching FreeCAD with the notebook run-copy macro...")
    print("Command:", [str(FREECAD_PATH), str(RUN_MACRO_PATH)])
    freecad_process = subprocess.Popen([str(FREECAD_PATH), str(RUN_MACRO_PATH)], cwd=str(WORKDIR))
    print("FreeCAD process started with PID:", freecad_process.pid)
    print("Wait for the FreeCAD window to finish generating the model; then run the status cell below.")

## 8. Check outputs after FreeCAD finishes

Run this cell after the FreeCAD GUI has had time to complete the macro. The `.FCStd` is the editable native FreeCAD document; the STL is only for in-notebook visualization.

In [ ]:
for p in (FCSTD_OUTPUT, STL_OUTPUT):
    if p.exists():
        print(f"FOUND  {p.name:40s} {p.stat().st_size:,} bytes")
    else:
        print(f"PENDING {p.name:40s} not present yet")
print("\nIf the file is still pending, check the FreeCAD GUI for errors and run this cell again.")

## 9. View the mesh exported by the executed macro

When the STL exists, this cell renders that mesh in Plotly. Otherwise it falls back to the neutral STEP preview from Section 4. The STL is an approximate tessellation suitable for visual inspection, not precision measurement.

In [ ]:
import plotly.graph_objects as go

def display_mesh(vertices_xyz, faces_ijk, title):
    xs, ys, zs = vertices_xyz[:, 0], vertices_xyz[:, 1], vertices_xyz[:, 2]
    fig = go.Figure(data=[go.Mesh3d(
        x=xs, y=ys, z=zs, i=faces_ijk[:, 0], j=faces_ijk[:, 1], k=faces_ijk[:, 2],
        opacity=1.0, flatshading=False,
        lighting=dict(ambient=0.55, diffuse=0.75, specular=0.2, roughness=0.7),
        name="K-EL/01")])
    fig.update_layout(title=title,
        scene=dict(xaxis_title="X (mm)", yaxis_title="Y (mm)", zaxis_title="Z (mm)", aspectmode="data"),
        margin=dict(l=0, r=0, t=45, b=0), height=700)
    fig.show()

if STL_OUTPUT.exists():
    import trimesh
    loaded = trimesh.load(str(STL_OUTPUT), force="mesh", process=False)
    if isinstance(loaded, trimesh.Scene):
        loaded = loaded.dump(concatenate=True)
    verts = loaded.vertices
    faces = loaded.faces
    display_mesh(verts, faces, "FreeCAD macro output — STL preview")
    print(f"Rendered STL with {len(verts):,} vertices and {len(faces):,} triangles.")
elif STEP_PATH is not None:
    print("Generated STL is not available yet; rerunning the STEP preview.")
    import cadquery as cq
    shape = cq.importers.importStep(str(STEP_PATH)).val()
    vv, tt = shape.tessellate(0.8, 0.35)
    import numpy as np
    vertices_xyz = np.array([[v.x, v.y, v.z] for v in vv], dtype=float)
    faces_ijk = np.array(tt, dtype=int)
    display_mesh(vertices_xyz, faces_ijk, "Supplied STEP concept preview")
else:
    print("Neither STL nor STEP is available. Confirm the file paths above.")

## Troubleshooting

- **`FreeCAD was not found`:** install FreeCAD on your local computer or set `FREECAD_EXECUTABLE` in Section 5 to the full path of the FreeCAD application executable. Restart/rerun Section 5.
- **Notebook is running on Colab:** Colab can display the STEP preview (with CadQuery/Plotly installed), but the hosted runtime normally cannot open the desktop FreeCAD GUI. Download the `.FCMacro` and run it on a computer with FreeCAD installed.
- **No `.FCStd` or STL appears:** check the FreeCAD window and its Python console/report view for exceptions. The macro prints a message when the document saves or when saving/export fails.
- **Mesh looks faceted:** decrease the `0.8` linear deflection in the STL export code and rerun the macro; this increases mesh size.
- **Remember:** geometry visualization is not engineering approval. The concept still needs dimensional/interface reconciliation, supplier confirmation, strength/duty/thermal analyses, and physical verification before any build.